## create a connection to the database using the library sqlite3

In [1]:
import pandas as pd
import sqlite3
connection = sqlite3.connect('../../data/checking-logs.sqlite')

In [2]:
connection.execute('DROP TABLE IF EXISTS datamart;')
query = '''
CREATE TABLE datamart AS
WITH wanted_commits AS (
    SELECT
        uid,
        labname,
        "timestamp" AS first_commit_ts
    FROM checker
    WHERE status = 'ready'
      AND numTrials = 1
      AND uid LIKE 'user_%'
      AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
)
SELECT
    fc.uid,
    fc.labname,
    fc.first_commit_ts,
    MIN(pv.datetime) AS first_view_ts
FROM wanted_commits fc
LEFT JOIN pageviews pv
    ON pv.uid = fc.uid
GROUP BY
    fc.uid,
    fc.labname,
    fc.first_commit_ts;'''
##pd.io.sql.read_sql(query, connection)
connection.execute(query)

In [3]:
datamart = pd.io.sql.read_sql('SELECT * FROM datamart;', connection)
datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


## using Pandas methods, create two dataframes: test and control

In [4]:
test = datamart[datamart['first_view_ts'].notnull()]
control = datamart[datamart['first_view_ts'].isnull()]

In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB


In [6]:
control.loc[:,'first_view_ts'] = test['first_view_ts'].mean()

In [7]:
control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [8]:
test.to_sql('test', connection, if_exists='replace', index=False)
control.to_sql('control', connection, if_exists='replace', index=False)

81

In [9]:
connection.close()